In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ann-notes/Comp_ANN_cheatsheet.pdf


### In this project we are building a Interview Question generator which will tak a pdf or notes as input and generate interview type questions and answers.

### Requirments:

* ### Langchain
* ### Gemini API


### Access the GeminiAPI key as Gemini is free

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GEMINI_KEY = user_secrets.get_secret("Gemini_key")


In [3]:
! pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 52.3 MB/s eta 0:00:00


In [4]:
!pip install langchain

In [5]:
!pip install -qU "langchain-chroma>=0.1.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.2/437.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.4/188.4 kB 9.1 MB/s eta 0

In [6]:
!pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Uninstalling protobuf-5.29.4:
      Successfully uninstalled protobuf-5.29.4
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.32.1 requires protobuf<6.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.16.1 requires protobuf<6.0.0dev,>=4.25.2; python_versi

In [7]:
!pip install langchain langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.6
    Uninstalling langchain-text-splitters-0.3.6:
      Successfully uninstalled langchain-text-splitters-0.3.6
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.18
    Uninstalling langchain-0.3.18:
      Successfully uninstalled langchain-0.3.18


### Import necessary libraries

In [8]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import TokenTextSplitter
from langchain.docstore.document import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain import embeddings
from langchain.prompts import PromptTemplate
from langchain.chains.summarize import load_summarize_chain
from langchain.vectorstores import FAISS

### Load pdf using pypdf loader

In [9]:
file_path = "/kaggle/input/ann-notes/Comp_ANN_cheatsheet.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

question_gen=""
for page in pages:
    question_gen+=page.page_content

### Create chunks 


In [10]:
# Method 1 for creating chunks
text_splitter=RecursiveCharacterTextSplitter(chunk_size = 10000, chunk_overlap= 200)
text=text_splitter.split_documents(pages)


# Method 2 for creating chunks need tiktoken tokenizer
text_splitter_token = TokenTextSplitter(
    chunk_size=10000,
    chunk_overlap=200)
texts_token = text_splitter.split_text(question_gen)



document_ques_gen=[Document(page_content=t) for t in texts_token]


doc_splitter_token = TokenTextSplitter(
    model_name="gpt-3.5-turbo",
    chunk_size=1000,
    chunk_overlap=200)
doc_token_chunks = text_splitter.split_documents(document_ques_gen)

### Create prompt template include refine prompt for a better output

In [11]:
prompt_template="""
You are an Expert Interviewer and today you are taked with creating a 
sets of only 10 interview question and answer from the text given below:
------------------
{text}
------------------------
The Question style will be as follows:
1. They should check the core knowledge of the interviewee
2. It should only be theory question and question about code will be askiing for a rough pseudo code/ steps.
3. The answer of the question should not be too big as the interview needs to be well timed.
4. The Questions need to take into account pre-requisites that one should know if they have studied the given document.
5. I just want question
Create questions that will prepare the coders or programmers for the interview make 
sure not to leave anything important.

Questions:
"""

refine_template = ("""
You are an expert at creating practice questions based on coding material and documentation.
Your goal is to help a coder or programmer prepare for a coding test.
We have received some practice questions to a certain extent: {existing_answer}.
We have the option to refine the existing questions or add new ones but keep the question count to 10 only
(only if necessary) with some more context below.
-----------
{text}
-----------

Given the new context, refine the original questions in English.
If the context is not helpful, please provide the original questions.
and in the end i dont want anything else just the questions no intro nothing
QUESTIONS:
""")

PROMPT_QUESTIONS=PromptTemplate(template=prompt_template, input_variables=['text'])

REFINE_QUESTIONS=PromptTemplate(template=refine_template, input_variables=['text', 'existing_answer'])

### Generate questions list

In [12]:
gemini_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-preview-04-17",
                 temperature=0.7, top_p=0.85, google_api_key=GEMINI_KEY)

ques_gen_chain=load_summarize_chain(llm=gemini_llm,
                                    chain_type='refine',
                                    verbose=True,
                                    question_prompt=PROMPT_QUESTIONS,
                                    refine_prompt=REFINE_QUESTIONS
                                    )

ques=ques_gen_chain.invoke(document_ques_gen)



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an Expert Interviewer and today you are taked with creating a 
sets of only 10 interview question and answer from the text given below:
------------------
Comprehensive ANN Cheat 
Sheet
Neural Network Fundamentals
Basic Components
Perceptron Single neuron that maps inputs to output via weights, bias, and 
activation
Forward propagation Input → weighted sum → activation → output
Backward propagation Calculate gradients with chain rule and update 
weights
Layer Collection of neurons processing information in parallel
Weights Learnable parameters that define strength of connections 
between neurons
Bias Learnable offset parameter for each neuron
Gradient Challenges
Problem Vanishing/exploding gradients
Solutions:
ReLU: f(x) = max(0,x)   Simple, fast, but "dying ReLU" problem
Leaky ReLU: f(x) = max(αx,x)  where α is small  Prevents dying neurons
ELU Exponential for 

In [13]:
ques

{'input_documents': [Document(metadata={}, page_content='Comprehensive ANN Cheat \nSheet\nNeural Network Fundamentals\nBasic Components\nPerceptron\ue092 Single neuron that maps inputs to output via weights, bias, and \nactivation\nForward propagation\ue092 Input → weighted sum → activation → output\nBackward propagation\ue092 Calculate gradients with chain rule and update \nweights\nLayer\ue092 Collection of neurons processing information in parallel\nWeights\ue092 Learnable parameters that define strength of connections \nbetween neurons\nBias\ue092 Learnable offset parameter for each neuron\nGradient Challenges\nProblem\ue092 Vanishing/exploding gradients\nSolutions:\nReLU: f(x) = max(0,x)  \ue088 Simple, fast, but "dying ReLU" problem\nLeaky ReLU: f(x) = max(αx,x)  where α is small \ue088 Prevents dying neurons\nELU\ue092 Exponential for negative values \ue088 Smoother gradient near zero\nWeight Initialization\ue092 Proper initialization \ue081He, Xavier) to maintain variance \nacr

In [14]:
print(ques['output_text'])

QUESTIONS:
1.  Based on the `create_model` function provided, explain the role of the `Dense` layers and the chosen activation functions (`relu` and `sigmoid`).
2.  Describe the purpose of the `Dropout` layer within the `create_model` function and how its `dropout_rate` parameter is used during hyperparameter tuning.
3.  Explain the parameters used when compiling the model in the `create_model` function, specifically mentioning the `optimizer='Adam'`, `loss='binary_crossentropy'`, and `metrics=['accuracy']`.
4.  What is the main reason for wrapping the Keras model in `KerasClassifier` when using Scikit-learn tools like `GridSearchCV` or `RandomizedSearchCV`?
5.  Compare the approaches of `GridSearchCV`, `RandomizedSearchCV`, and Bayesian Optimization (using Optuna) for hyperparameter tuning, highlighting a key difference between Grid Search and Randomized Search.
6.  Explain the purpose and configuration of the `EarlyStopping` callback as shown in the examples.
7.  Describe the functio

### Now for answers store the documents in the vector store

In [15]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings


gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GEMINI_KEY)

vector_store = FAISS.from_documents(doc_token_chunks, gemini_embeddings)



### Generate answers and save them in txt file

In [16]:
from langchain.chains import RetrievalQA


ques_list = [q for q in ques['output_text'].split("\n") if q.strip() and not q.strip() == "QUESTIONS:"]

answer_generation_chain = RetrievalQA.from_chain_type(llm=gemini_llm,
                                               chain_type="stuff",
                                               retriever=vector_store.as_retriever())

for question in ques_list:
    print("Question: ", question)
    answer_result = answer_generation_chain.run(question)
    
    # Check if answer is a dictionary and extract the actual answer
    if isinstance(answer_result, dict):
        if 'result' in answer_result:
            answer = answer_result['result']
        else:
            # Use str() as fallback if we can't extract a specific field
            answer = str(answer_result)
    else:
        answer = answer_result
        
    print("Answer: ", answer)
    print("--------------------------------------------------\n\n")
    
    # Save answer to file
    with open("answers.txt", "a", encoding="utf-8") as f:
        f.write("Question: " + question + "\n")
        f.write("Answer: " + answer + "\n")
        f.write("--------------------------------------------------\n\n")

Question:  1.  Based on the `create_model` function provided, explain the role of the `Dense` layers and the chosen activation functions (`relu` and `sigmoid`).


/tmp/ipykernel_13/1756618401.py:12: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  answer_result = answer_generation_chain.run(question)


Answer:  Based on the `create_model` function and the provided context:

1.  **Dense Layers:**
    *   The `Dense` layers are the fundamental building blocks of this simple neural network.
    *   Each `Dense` layer is a "Collection of neurons processing information in parallel".
    *   Within each neuron in a `Dense` layer, the input from the previous layer is processed via a "weighted sum" using "Weights" (learnable parameters defining connection strength) and a "Bias" (a learnable offset).
    *   The result of this weighted sum plus bias is then passed through an "activation" function.
    *   The first `Dense` layer in the function has `units` number of neurons and defines the `input_shape` for the network.
    *   The subsequent `Dense` layer (the output layer) has 1 neuron, appropriate for a binary classification task.

2.  **Activation Functions:**
    *   **`relu` (Rectified Linear Unit):**
        *   Used in the first `Dense` layer, which acts as a hidden layer.
        *  